# 0. Radon Thesis

Matthew Thompson, fall 2026, EWU. 

## 0.1 Abstract

In this paper we investigate the basic idea of the Radon transform and its use in tomographic reconstruction. We consider the filtered back projection method of inversion, deriving it from the Fourier slice theorem. We analyze the numerical attributes of various common choices of filter kernel. We construct a filter kernel which is a signed Gaussian mixture by using machine learning. We train a nerual network which plays the role of a filter kernel. Finally, we give a brief discussion of generalized Radon transforms and the theory of Radon transforms.  

## 0.2 Contents

1. Definitions and introduction
   1. Introduction
   2. Radon Transform
   3. Filtered Back Projection
2. Theory of Radon transform in R^2
   1. Fourier slice theorem
   2. Mapping of schwartz space, plancherel's formula
   3. Inversion
   4. Derivation of FBP
   5. FBP range, injectivity results (proof sketches only)
   6. A brief mention of sampling theory
3. Techniques in Parallel beam X-Ray tomography
   1. Literature review, list of filters
   2. Noise
   3. Numerical analysis
4. Learning of filter kernel
   1. Difference of gaussians filter window, least squares regression
   2. CNN filter window
   3. In-between approach, assessing noise and selecting from a family of filters
5. (Can omit) A brief discussion of further topics
   1. Radon transform in higher dimensions
   2. Reconstruction of probability distributions, cramer-wold theorem
   3. Support theorem
   4. Distriubtions
   5. Generalized Radon transform, a few examples
   6. more on the Radon transform (homogeneous spaces, double fibration)

# 1. Definitions, basic ideas, and introduction

## 1.1 Introduction

Inverse problems involve solving $f(x) = y$ for $x$, given $f$ and $y$. We can think of this as reconstructing $x$ from $y=f(x)$, or as finding the inverse function $f^{-1}$ and applying it to $f^{-1}(y) = x$. 

Computed Tomography is such an inverse problem. 

#### Physical scenario of Tomography

Radon inversion is used in medical imaging to reconstruct images from sensor data. X-rays passing through matter experience attenuation according to known laws: $I = I_0 e^{-Al}$. Thus we have a way to physically measure the line integrals in the Radon transform. Collecting sensor data from all angles and a range of displacements gives the full radon transform data. The transform of the density function in the Radon domain is called a sinogram. 

(In real applications, fan-beam and limited angle tomography is more common. But this paper will focus on parallel beam x-ray.)

## 1.2 Radon Transform

####  Radon transform

The Radon transform maps a function $f$ defined on $\mathbb{R}^n$ to a function defined over the set of all $n-1$ dimensional hyperplanes in $\mathbb{R}^n$. 

\begin{align*} \mathcal{R}f (r, \alpha) = \int_{\langle x, \alpha \rangle = r} f(x) dx \end{align*}




For the case of $\mathbb{R}^2$, given a function $f: \mathbb{R}^2 \to \mathbb{R}$, and the lines $\Gamma(r, a) = \{x, y \in \mathbb{R}^2 | r = x \cos(a) + y \sin(a)\}$, we define the radon transform as 

\begin{align*} \mathcal{R}f (r, a) = \int_{-\infty}^{\infty}    f(r\cos(a) - t \sin(a), r\sin(a) + t \cos(a)  )     dt. \end{align*}






We say a point $(x, y)$  is incident with the line $\Gamma(r, a)$ if they "touch", that is if $(x,y) = (r\cos(a) - t \sin(a), r\sin(a) + t \cos(a)$ for some $t$. We can capture this idea of incidence by writing the Radon transform in terms of the Dirac delta function.

\begin{align*} \mathcal{R}f (r, a) = \int \int_{\mathbb{R}^2} f(x, y) \delta (r - x \cos(a) - y \sin(a)) dx dy \end{align*}

We can think of $\mathcal{R}f$ as a function on the tangent bundle to $S^1$, since each point (line) in the Radon transformation space is associated with the angle $a$ perpendicular to it, as well as the distance from the origin of point on the line which intersects the line $(x,y) = (t \cos(a), t \sin(y))$. (SOURCE: Natterer)



#### Adjoint

We can also consider the dual to the radon transform, $\mathcal{R}^*$ which sends each line of the function $\mathcal{R}f(r, \alpha)$ back to a point $x \in \mathbb{R}^n$. If $g(\gamma) = \mathcal{R}f(x)$, then 

\begin{align*} \mathcal{R}^* g (x) = \int_{x \in \gamma} g(\gamma) d\mu(\gamma), \end{align*}

where $d\mu(\gamma)$ is a certain "unique probability measure on $\{\gamma | x \in \gamma\}$". (SOURCE: WIKIPEDIA)

Later on we might define more explicitly what this measure is, and use it to define the Radon transform on a generalized function $T$, by using a test function $g$ and duality: $\langle \mathcal{R}T , g \rangle = \langle T, \mathcal{R}^* g \rangle$.

In the case of $\mathbb{R}^2$, the dual radon transform is 

\begin{align*}  \end{align*}




## 1.3 Filtered Back Projection

#### Issues with simple back projection

Consider 

\begin{align*} \mathcal{R}^*(\mathcal{R}f(a)) = \end{align*}

Simply applying the adjoint radon transform to a sinogram fails to invert the Radon transform. The projection data is simply smeared back accross the spatial domain, which results in blurring and star shaped artifacts. In effect, the back projection acts as a low pass filter (WHY IS THIS? WHAT IS THE FREQUENCY CUTOFF?) and obscures details of the image which correspond to high frequencies in the Fourier domain, such as edges and fine-grained patterns.  

#### Filtered Back Projection

To avoid this, a ramp filter $|\omega|$ is introduced, which amplifies higher frequencies in the fourier domain, and subdues the low-frequency blurring.

However, introduction of this high-frequency-amplifying ramp filter makes the filtered back projection extremely sensitive to high-frequency noise, and ill-posed. 

So we introduce additional filtering $|\omega|W(\omega)$, often in the form of a window which removes the highest frequencies, while still keeping the overall ramp shape. Note that $W(\omega)$ is a weight function, which approximates the identity in some sense. 

Examples of filters are Ram-lak, shepp-philips, Lam, and others. We will first investigate very basic filters which are not effective.

## 2. Theory of Radon transforms in $\mathbb{R}^2$

## 2.1 Fourier Slice theorem

#### Fourier inversion theorem

For a continuous function $f$, we define the Fourier transform 

\begin{align*} (\mathcal{F}f)(\xi) = \int_{-\infty}^{\infty} e^{-2i \pi y \xi} f(y) dy. \end{align*}

For functions $f$ satisfying certain conditions, and denoting $g(\xi) = \mathcal{F}f(\xi)$, the inverse Fourier transform is defined as

\begin{align*} (\mathcal{F}^{-1}g)(x) = \int_{-\infty}^{\infty} e^{2i \pi x \xi}g(\xi) d\xi. \end{align*}

(PROOF FOR SCHWARTZ FUNCTIONS?)

Taken together, the Fourier inversion theorem states that, for functions $f$ satisfying certain conditions, $f$ can be reconstruced from its fourier transform by applying the inverse fourier transform, that is, 

\begin{align*} f(x) &= \int_{-\infty}^{\infty} e^{2i \pi x \xi} \left( \int_{-\infty}^{\infty} e^{-2i \pi y \xi} f(y) dy \right)d\xi. \\
&=  \int_{-\infty}^{\infty} \int_{-\infty}^{\infty} e^{2i \pi (x - y) \xi}f(y) dy d\xi. \end{align*}  

One such satisfactory condition would be that $f$ belongs to the Schwartz space.

## 2.2 Mapping of Schwartz space

For our purposes, we will assume that the function we wish to construct is rapidly decreasing. That is, it is in the Schwartz space 

\begin{align*} \mathcal{S}(\mathbb{R}^2) = \{f \in |  \} \end{align*}

## 2.3 Inversion of Radon transform

#### Ludwig's inversion formula, Hilbert transform

#### (Can discuss Nievergelt's elementary inversion method)

#### Riesz potential

#### Filtered back projection

# 3. Techniques in Parallel beam X-Ray tomography

## 3.1 Literature review, list of filters

#### Ram-Lakk, or Ramp filter

The Ram-Lakk filter is just the ramp filter with no transfer function, and is the starting point for all of the other filters. It is $|\omega|$, where $\omega$ is in the frequency domain resulting from taking the fourier transform along the detector axis of each view angle in the sinogram..

## 3.2 Noise

Two types of noise commonly occur in X-ray imaging scenarios.

#### Poisson noise

Variability in photon counts due to quantum... stuff. Follows a Poisson distribution. Also called shot noise. The photon count of each detector is not a true count of the expected photons according to the initial intesity $I_0$ and the attenuation, but rather selected from a Poisson distribution with mean of the true expected count, and variance proportional to the amount of attenuation as well as $I_0$.

\begin{align*}


\end{align*}

#### Gaussian noise



\begin{align*}


\end{align*}

## 3.3 Sampling

Since the number of sensors and the time-steps of each sensor are discrete, it is necessary to deal not with the full Radon transform of the continuous density function, but rather samples of the transform. 


#### Band limited functions

"A function is considered band limited with bandwidth $b$ if its Fourier transform is locally integrable and vanishes outside the ball with radius $b$." (SOURCE: Natterer)

Shannon's sampling theorem: Let $f$ be $b$-bandwidth limited, and let $h \leq \pi / b$.Then $f$ is uniquely determined by the values $f(hk)$ for $k \in \mathbb{Z}^n$, and in $L_2(\mathbb{R}^n)$, (mean square convergence), we have 

\begin{align*} f(x) = \sum_{k} f(hk) \operatorname{sinc} \frac{\pi}{h}(x-hk). \end{align*}

and also, in $L_2([-\pi/h, \pi/h]^n)$, 

\begin{align*} \hat{f}(\xi) = (2\pi)^{-n/2}h^n \sum_{k} f(hk)e^{-ih \xi \cdot k}.\end{align*}

If $g$ is another $b$-band limited function, then 

\begin{align*} \int_{\mathbb{R}^n} f(x) \overline{g}(x) dx = h^n \sum_{k} f(hk) \overline{g}(hk). \end{align*}

Proof: 

#### Nyquist

$h \leq \pi/b$ is the Nyquist condition. When $f$ is sampled at less than half the smallest detail contained at $f$, (when Nyquist is satisfied), $f$ can be reconstructed from its samples by sinc series, and this reconstruction is stable in some sense. (SOURCE: Natterer) From the third equation, 

\begin{align*} ||f||_{L_2(\mathbb{R}^n)} = \end{align*}

Note that sinc functions are hard to compute numerically. (so maybe we can use b-spline or difference-of-gaussians interpolation?)

This is the case for image reconstruction from direct samples of the image, but what of reconstruction from samples of the radon transform? 

# 4. Learning of Filter 

## 4.1 Least squares to train a ~~difference-of-gaussians~~ filter window $W(\omega)$

The filter kernel we use in FBP will be $H(\omega) = |\omega|W(\omega)$, where $|\omega|$ is the standard Ramm-lak ramp filter. We want to use machine learning to produce a filter window $W(\omega)$ which is effective at reducing the types of noise common to CT applications, (while of course preserving the reconstruction capability of Ramm-lak.)

We will generate a dataset consisting of paired ellipse-phantoms and noised sinograms. We will use various mixtures of the two types of noise common to real-world CT applications, namely Poisson and Gaussian. 

~~The windowing function will be initialized as a sum of Gaussians with set means and variances. The paramters to be altered will be the weight of each Gaussian (allowing negative weights.) For example, a grid of 400 gaussians with 20 different means and 20 different variances. We will use least squares regression to optimize the performance of $W(\omega)$ as a filter window during FBP on the given dataset.~~ 

~~In order for the regression to be effective, and not just get lost and directionless in the ~400 degrees of freedom, we shall perhaps have to use some kind of regularization, so that nearby-gaussians can supress eachother and we get appropriately distinctive spiking.~~

The window $W$ should just be a continuous function. (sampled from discretely during the finite implementation of FBP.) Initialize it as one of the standard filter-windows. 

The MSE loss function should be replaced with a more sophisticated combination of error measures, so that there is no MSE-specific bias. 

The learned filter will be optimized to have the average best performance accross a range of different noise paramaters.

Consider having $W$ act on the normalized frequency, $W(\omega / max(|\omega|)), so that the same filter can be applied across the entire frequency range of sinograms with different max frequencies. 



## ~~4.2 Training a noise-assessment neural network $A(\phi_1; \mathcal{R}f) = \theta$, and a filter-selector function $G(\phi_2; \theta) = W_{\theta}$.~~

**(MOSTLY DISCARDING THIS IDEA, DITCHING THE DIFFERENCE-OF-GAUSSINS THING, AND INSTEAD MOVING TOWARD A COMBINATION OF THE IDEAS FROM 4.1, IMPLEMENTED AS A CNN AS IN 4.3.)**

~~So the back half $G$ would be trained on a finite dataset of noise paramaters $\theta$, paired with respective filter kernel (paramaters) $W_\theta$ for that given noise paramater (which we construct beforehand using the least squares regression method from earlier, training on a dataset of phantoms noised with that particular noise paramater.) So that given a general noise paramater $\hat{\theta}$, $G$ can predict a least-squares-optimizing filter (paramaters) $W_{\hat{\theta}}$.~~

~~Also, for $G$, instead of optimizing the discrete set of 400 paramaters, it may be better to optimize a continuous function which is somehow sampled (perhaps by FFT) to give the 400 paramaters.~~

~~And then the front half $A$ would be trained on a dataset of noised sinograms paired with their respective nosie paramaters $\theta$, so that given a general noised sinogram, it can predict the noise paramater $\hat{\theta}$ which would produce such a noised sinogram.~~

~~Thus, running these end-to-end and then performing FBP using the chosen kernel, we should be able to input a noised sinogram and get out its reconstruction using FBP with an optimally-noise-reducing kernel.~~

~~(And in this instance we are abusing notation to have $W_{\theta}$ refer to both the difference-of-gaussians function $W_\theta(\omega)$, as well as the respective paramaterization (weighting of the Gaussians).)~~

~~$A(\mathcal{R}f)$ should be a neural network, specifically a CNN, so that it can deal with the higher-level structure and connecteivity of the whole sinogram $\mathcal{R}f$, and not just the individual attenuation values at each point (line) in $\mathcal{R}f$.~~

~~$G(\theta)$ could be a neural network, or just a least squares regression model. which learns on a dataset of paired noise mixtures $\theta$ and optimal filter paramterizations $W_\theta$.~~ 

~~The dataset of $W_\theta$ will be constructed by training on a dataset of paired ground-truth phantoms and sinograms noised with noise paramter $\theta$, using the least-squares regression method from section 4.1.~~

## 4.3 training a CNN $W(\theta; \omega, \mathcal{R}f)$

Perhaps there would be an advantage to merging the two halves from section 4.2 even more closely, so that the input is a noised sinogram and the output is an optimal filter which is then used to apply FBP, with no separation of the in-between steps.

# 5. A brief discussion of further topics

## 5.1 Radon transform in $\mathbb{R}^3$ and $\mathbb{R}^n$

## 5.3 Radon transform of Tempered Distributions

Dissributions are generalized functions, which are linear functionals acting on a space of test functions. For example, one family of distributions can be defined by taking the space of functions to be the functions with compact support, $D(\mathbb{R})$.

So distributions are linear operators $T: D(\mathbb{R}) \to \mathbb{R}$ which act on functions $\phi$. Instead of writing $T(\phi)$, we write $\langle T, \phi \rangle$ to denote the distribution $T$ evaluated at the test function $\phi$.

For example, the Dirac delta $\delta$ can be defined as 

\begin{align*} \langle \delta , \phi \rangle = \phi(0). \end{align*}

(SOURCE: Wikpiedia) "Derivatives of generalized functions are defined via transposition to the test functions using integration by parts

\begin{align*}  \langle f', \phi \rangle = \int_{-\infty}^{\infty} f' \phi dx
= \bigl[f(x)\phi(x) \bigr]_{-\infty}^{\infty} - \int_{-\infty}^{\infty} f' \phi dx
= - \langle f, \phi' \rangle,\end{align*} 

(Note that $\bigl[f(x)\phi(x) \bigr]_{-\infty}^{\infty} = 0$. Since $\phi \in D(\mathbb{R})$, it is equal to zero outside of some bounded interval.) 

allowing differentiation of non-differentiable or discontinuous functions (like the Heaviside step function or Dirac delta function)." 

The tempered distributions are the generalized functions which take the Schwartz space as their space of test functions. 

The theory we have been discussing for Radon transforms of functions in the schwartz space also holds for tempered distributions. 

#### Fourier inversion


